<a href="https://colab.research.google.com/github/Jahdejosefina/projecto_ragTP2/blob/main/00_corpus_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive to Server...' from Ctrl+Shift+P, then re-run this cell)
Mounted at /content/drive
artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
Colab bootstrap OK. Working directory: /content/projecto_rag


# Etapa 0 · El corpus como decisión
Este notebook presenta la auditoría inicial del corpus de legislación nacional argentina, obtenido del repositorio legalize-ar y limitado a 170 documentos relacionados con defensa del consumidor.

El objetivo es evaluar su tamaño, estructura, calidad textual, metadatos, identificadores y similitud entre documentos antes de construir el sistema RAG.

También se realiza un chequeo a libro cerrado con el generador Qwen3-8B para explorar su conocimiento previo sobre la temática.

In [36]:
import environment  # noqa: F401  (efectos secundarios: guarda de JAX, caché de HF, bundle TLS, UTF-8)

environment.summary()

from console import StageRun, clip, print_table, verdict

SMOKE_TEST = False  # True: unas pocas páginas, minutos en una laptop. False: el subconjunto entero.
SEED = 1337
ART = environment.artifacts_dir(SMOKE_TEST)
PROFILE = "smoke" if SMOKE_TEST else "full"
run = StageRun.start("00_corpus_audit", ART, profile=PROFILE, config={"seed": SEED})

ENVIRONMENT
os             : Linux 6.6.122+
python         : 3.13.15
interpreter    : /usr/bin/python3
colab          : True
logical_cores  : 2
torch          : 2.11.0+cu130
cuda_available : True
gpu            : Tesla T4
gpu_memory_gb  : 15.64
ram_total_gb   : 13.6
ram_free_gb    : 11.2
HF_HOME        : /content/drive/MyDrive/projecto_rag/hf_cache
jax_guard      : True
ca_bundle      : not present (normal)
artifacts      : -> /content/drive/MyDrive/projecto_rag/artifacts (Drive)
== 00_corpus_audit · profile full · report -> /content/projecto_rag/artifacts/full/run_report.json


In [47]:
import store
import os

# 1. El repositorio de las leyes argentinas sugerido en corpora.md
LEGAL_REPO = "https://github.com/legalize-dev/legalize-ar"

corpora = environment.corpora_dir()
legal_root = corpora / "legalize-ar"

# Descargamos el repositorio en un commit fijo.
# Esto permite trabajar sobre una versión reproducible del corpus.

store.fetch_git(LEGAL_REPO, legal_root, branch="main", commit="3d2450b")

# Verificamos que se haya descargado y contamos los archivos
md_files = list(legal_root.rglob("*.md"))
print(f"Repositorio clonado en: {legal_root}")
print(f"Cantidad de archivos Markdown (normas) encontrados: {len(md_files)}")

reusing legalize-ar @ 3d2450b5c09a
Repositorio clonado en: /content/projecto_rag/corpora/legalize-ar
Cantidad de archivos Markdown (normas) encontrados: 31327


### El tema elegido: Defensa del consumidor (Ley 24.240 y modificatorias)

In [38]:
palabras_clave = ["defensa del consumidor", "24.240", "ley 24240"]

subconjunto = []
for path in md_files:
    # Leemos el texto en minúsculas para facilitar la búsqueda
    texto = path.read_text(encoding="utf-8").lower()

    # Si alguna de las palabras clave está en el texto de la ley, la guardamos
    if any(keyword in texto for keyword in palabras_clave):
        subconjunto.append(path)

print(f"Archivos encontrados para Defensa del Consumidor: {len(subconjunto)}")

# Imprimimos los nombres de los primeros 10 archivos para verificar qué agarró
print("\nPrimeros 10 archivos del subconjunto:")
for path in subconjunto[:10]:
    print(" -", path.name)

Archivos encontrados para Defensa del Consumidor: 170

Primeros 10 archivos del subconjunto:
 - DNU-271-2000.md
 - DEC-946-1989.md
 - LEY-27250.md
 - DNU-13-2015.md
 - DNU-1283-2003.md
 - DEC-991-2020.md
 - DEC-528-2011.md
 - DEC-276-1998.md
 - DNU-2156-2002.md
 - DEC-826-2011.md


## ¿Hace falta siquiera la recuperación?

La línea de base honesta para cualquier sistema de RAG es: *si entra, pegalo*. Entonces: ¿qué
tan grande es este corpus en los tokens del modelo que lo va a leer -- no en caracteres, no en
palabras, y no en los tokens del embedder (cada modelo divide el mismo texto de forma distinta)?

In [39]:
import re
import llm
import statistics
import pandas as pd

GENERATOR = "qwen3:8b"
NUM_CTX = 8192

fechas_malas = 0
encabezados_encontrados = set()
total_chars = 0
resultados_tokens = []
errores_tokens = []

for path in subconjunto:
    texto = path.read_text(encoding="utf-8")
    total_chars += len(texto)

    # 1. Chequear metadatos engañosos
    if 'last_updated: "1900-01-01"' in texto:
        fechas_malas += 1

    # 2. Buscar variaciones tipográficas de "Artículo"
    matches = re.findall(
        r'(?i)(#*\s*art[íi]culo\s+\d+[º°]?\s*[—\-\.]*)',
        texto
    )

    for m in matches[:2]:
        encabezados_encontrados.add(m.strip())

    # 3. Contar tokens de cada documento por separado
    if not texto.strip():
        resultados_tokens.append({
            "documento": path.name,
            "caracteres": len(texto),
            "tokens": None,
            "estado": "vacío"
        })
        continue

    try:
        cantidad_tokens = llm.count_tokens(texto, GENERATOR)

        resultados_tokens.append({
            "documento": path.name,
            "caracteres": len(texto),
            "tokens": cantidad_tokens,
            "estado": "OK"
        })

    except Exception as e:
        errores_tokens.append({
            "documento": path.name,
            "error": str(e)
        })

        resultados_tokens.append({
            "documento": path.name,
            "caracteres": len(texto),
            "tokens": None,
            "estado": "error"
        })


# 4. Preparar la tabla de resultados
df_tokens = pd.DataFrame(resultados_tokens)

df_validos = df_tokens[
    (df_tokens["estado"] == "OK")
    & (df_tokens["tokens"].notna())
]

print("--- RESULTADOS DE LA AUDITORÍA DE DEFENSA DEL CONSUMIDOR ---")
print(f"Archivos analizados: {len(subconjunto)}")
print(
    "Leyes con fecha 'last_updated' falsa (1900-01-01):",
    fechas_malas
)
print(
    "Ejemplos de cómo escriben 'Artículo':",
    list(encabezados_encontrados)[:5]
)

print(f"\nCaracteres totales del subconjunto: {total_chars:,}")
print(f"Documentos con conteo válido de tokens: {len(df_validos)}")
print(f"Documentos vacíos: {(df_tokens['estado'] == 'vacío').sum()}")
print(f"Documentos con errores: {len(errores_tokens)}")

# 5. Distribución de tokens
if not df_validos.empty:
    tokens = df_validos["tokens"]

    print("\n--- DISTRIBUCIÓN DE TOKENS POR DOCUMENTO ---")
    print(f"Tokens totales contados: {tokens.sum():,}")
    print(f"Promedio por documento: {tokens.mean():,.1f}")
    print(f"Mediana por documento: {tokens.median():,.1f}")
    print(f"Mínimo: {tokens.min():,}")
    print(f"Máximo: {tokens.max():,}")
    print(f"Percentil 25: {tokens.quantile(0.25):,.1f}")
    print(f"Percentil 75: {tokens.quantile(0.75):,.1f}")
    print(f"Percentil 90: {tokens.quantile(0.90):,.1f}")

    print(
        f"\nEquivalencia orientativa: "
        f"{tokens.sum() / NUM_CTX:.1f} ventanas de {NUM_CTX} tokens."
    )

    print("\nDocumentos más largos:")
    display(
        df_validos.nlargest(10, "tokens")[
            ["documento", "caracteres", "tokens"]
        ]
    )

    print("\nDocumentos más cortos:")
    display(
        df_validos.nsmallest(10, "tokens")[
            ["documento", "caracteres", "tokens"]
        ]
    )

if errores_tokens:
    print("\n--- ERRORES DE CONTEO ---")
    display(pd.DataFrame(errores_tokens))

--- RESULTADOS DE LA AUDITORÍA DE DEFENSA DEL CONSUMIDOR ---
Archivos analizados: 170
Leyes con fecha 'last_updated' falsa (1900-01-01): 159
Ejemplos de cómo escriben 'Artículo': ['artículo 6º', '##### Artículo 1º -', 'ARTICULO 155', 'ARTICULO 23', 'ARTICULO 13.']

Caracteres totales del subconjunto: 10,835,049
Documentos con conteo válido de tokens: 170
Documentos vacíos: 0
Documentos con errores: 0

--- DISTRIBUCIÓN DE TOKENS POR DOCUMENTO ---
Tokens totales contados: 3,115,531
Promedio por documento: 18,326.7
Mediana por documento: 6,378.5
Mínimo: 512
Máximo: 341,824
Percentil 25: 2,532.0
Percentil 75: 17,512.8
Percentil 90: 45,436.8

Equivalencia orientativa: 380.3 ventanas de 8192 tokens.

Documentos más largos:


,documento,caracteres,tokens
13,LEY-26994.md,1221802,341824
30,DEC-50-2019.md,829748,252875
128,DEC-958-2018.md,383021,105058
137,DEC-357-2002.md,263230,103316
49,DEC-1359-2004.md,292627,85321
32,LEY-26522.md,306078,84833
166,DEC-92-1997.md,272285,80317
161,LEY-26221.md,274122,77608
16,DEC-764-2024.md,272489,75862
122,DNU-805-2025.md,266687,75234



Documentos más cortos:


,documento,caracteres,tokens
65,DEC-939-2016.md,1225,512
75,DEC-938-2016.md,1225,512
23,DEC-765-2002.md,1299,517
153,DEC-1442-2002.md,1329,526
155,DEC-436-2002.md,1346,536
107,DEC-433-2001.md,1382,544
79,DEC-420-2006.md,1402,568
48,DEC-875-2008.md,1427,572
64,DEC-79-2003.md,1460,579
89,DEC-901-2002.md,1628,619


In [40]:
import re
import random
import itertools
import statistics
import llm

# 1. DISTRIBUCIÓN DE LARGOS EN TOKENS
# Reutilizamos los conteos reales de la celda anterior.
largos_tokens = df_validos["tokens"].astype(int).tolist()

print("--- 1. DISTRIBUCIÓN DE LARGOS ---")
print(f"Documentos contados: {len(largos_tokens)}")
print(f"Largo promedio: {statistics.mean(largos_tokens):,.0f} tokens")
print(f"Mediana: {statistics.median(largos_tokens):,.0f} tokens")
print(f"Documento más largo: {max(largos_tokens):,} tokens")
print(f"Documento más corto: {min(largos_tokens):,} tokens")



# 2. CASI-DUPLICADOS: JACCARD DE SHINGLES


# Usamos una semilla fija para poder repetir el experimento.
rng = random.Random(42)

muestra = rng.sample(
    sorted(subconjunto, key=lambda p: str(p)),
    min(40, len(subconjunto))
)

def crear_shingles(texto, n=5):
    """
    Convierte el texto en conjuntos de secuencias
    de 5 palabras consecutivas.
    """
    palabras = re.findall(r"\w+", texto.lower())

    if len(palabras) < n:
        return set()

    return {
        tuple(palabras[i:i+n])
        for i in range(len(palabras) - n + 1)
    }


def similitud_jaccard(a, b):
    """
    Calcula la similitud entre dos conjuntos:
    elementos compartidos / elementos totales.
    """
    union = a | b

    if not union:
        return 0.0

    return len(a & b) / len(union)


# Generamos los shingles de cada documento una sola vez.
shingles_documentos = {}

for path in muestra:
    texto = path.read_text(encoding="utf-8")
    shingles_documentos[path.name] = crear_shingles(texto)


# Comparamos todos los pares de la muestra.
similitudes = []

for doc1, doc2 in itertools.combinations(
    shingles_documentos.keys(), 2
):
    score = similitud_jaccard(
        shingles_documentos[doc1],
        shingles_documentos[doc2]
    )

    similitudes.append((doc1, doc2, score))


# Ordenamos de mayor a menor similitud.
similitudes.sort(key=lambda x: x[2], reverse=True)

# Umbral exploratorio, no una definición universal.
UMBRAL = 0.80

pares_similares = [
    par for par in similitudes
    if par[2] >= UMBRAL
]

print("\n--- 2. CASI-DUPLICADOS ---")
print(f"Documentos en la muestra: {len(muestra)}")
print(f"Pares comparados: {len(similitudes)}")
print(f"Pares con Jaccard >= {UMBRAL}: {len(pares_similares)}")

print("\nLos 10 pares con mayor similitud:")

for doc1, doc2, score in similitudes[:10]:
    print(f"{doc1} <-> {doc2}: {score:.3f}")



# 3. CHEQUEO A LIBRO CERRADO


print("\n--- 3. CHEQUEO A LIBRO CERRADO ---")

preguntas_libro_cerrado = [
    "¿Qué dice el artículo 4 de la Ley 24.240 de Defensa del Consumidor sobre la información?",
    "¿Cuáles son las sanciones principales previstas en la Ley de Defensa del Consumidor en Argentina?",
    "¿Qué es el daño punitivo en el derecho del consumidor argentino?",
    "¿Qué plazo hay para reclamar por vicios redhibitorios según la Ley 24.240?",
    "¿Es válido un contrato de consumo que renuncia a los derechos del consumidor en Argentina?"
]

llm.setup(GENERATOR, num_ctx=NUM_CTX)

if llm.server_alive() and llm.model_digest(GENERATOR):
    for q in preguntas_libro_cerrado:
        r = llm.ask(
            q,
            system="Respondé en una oración.",
            model=GENERATOR,
            num_ctx=NUM_CTX,
            num_predict=160,
            cache=False
        )

        print(
            f"Pregunta: {q}\n"
            f"Respuesta del modelo: {r.text}\n"
        )
else:
    print(
        "El servidor de Ollama no está respondiendo "
        "para hacer el test a libro cerrado."
    )

--- 1. DISTRIBUCIÓN DE LARGOS ---
Documentos contados: 170
Largo promedio: 18,327 tokens
Mediana: 6,378 tokens
Documento más largo: 341,824 tokens
Documento más corto: 512 tokens

--- 2. CASI-DUPLICADOS ---
Documentos en la muestra: 40
Pares comparados: 780
Pares con Jaccard >= 0.8: 0

Los 10 pares con mayor similitud:
LEY-24787.md <-> DEC-270-1997.md: 0.482
DEC-960-2017.md <-> DEC-788-2003.md: 0.325
DEC-507-1994.md <-> DEC-2453-1993.md: 0.286
LEY-27266.md <-> LEY-27265.md: 0.219
DEC-404-2022.md <-> DEC-764-2024.md: 0.180
LEY-27250.md <-> LEY-27265.md: 0.163
LEY-27266.md <-> LEY-27250.md: 0.135
LEY-25093.md <-> LEY-27265.md: 0.099
DEC-866-2025.md <-> DEC-764-2024.md: 0.094
LEY-27250.md <-> LEY-25093.md: 0.084

--- 3. CHEQUEO A LIBRO CERRADO ---
   model        qwen3:8b
   digest       500a1f067a9f
   server       0.34.4
   num_ctx      8192
   thinking     {'honors_think_false': True, 'leaks_think_tag': False, 'thinking_only': False, 'supports_thinking': True, 'answer': 'Paris'}
   key

In [41]:
import random

# Selección reproducible de 10 documentos
rng = random.Random(42)
documentos_revision = rng.sample(
    sorted(subconjunto, key=lambda p: str(p)),
    10
)

print("--- DOCUMENTOS SELECCIONADOS PARA REVISIÓN MANUAL ---")

for i, path in enumerate(documentos_revision):
    print(f"{i}. {path.name}")

def mostrar_documento(i):
    path = documentos_revision[i]
    texto = path.read_text(encoding="utf-8")

    print(f"DOCUMENTO: {path.name}")
    print(f"CARACTERES: {len(texto)}")
    print("=" * 60)
    print(texto)

--- DOCUMENTOS SELECCIONADOS PARA REVISIÓN MANUAL ---
0. LEY-27266.md
1. DEC-21-2007.md
2. DEC-1225-2010.md
3. DEC-507-1994.md
4. DEC-433-2001.md
5. DEC-405-1998.md
6. DEC-2453-1993.md
7. DEC-2089-1993.md
8. LEY-24787.md
9. DEC-196-2025.md


In [42]:
mostrar_documento(0)
mostrar_documento(1)
mostrar_documento(2)
mostrar_documento(3)
mostrar_documento(4)
mostrar_documento(5)
mostrar_documento(6)
mostrar_documento(7)
mostrar_documento(8)
mostrar_documento(9)


DOCUMENTO: LEY-27266.md
CARACTERES: 2509
---
title: "DEFENSA DEL CONSUMIDOR"
identifier: "LEY-27266"
country: "ar"
rank: "ley"
publication_date: "2016-08-17"
last_updated: "1900-01-01"
status: "in_force"
source: "http://servicios.infoleg.gob.ar/infolegInternet/anexos/260000-264999/264485/norma.htm"
department: "HONORABLE CONGRESO DE LA NACION ARGENTINA"
infoleg_id: "264485"
enactment_date: "2016-07-13"
gazette_number: "33441"
gazette_page: "1"
texto_resumido: "SUSTITUYESE EL ARTICULO 38 DE LA LEY 24.240."
original_text_url: "http://servicios.infoleg.gob.ar/infolegInternet/anexos/260000-264999/264485/norma.htm"
times_modified: "1"
modifies_count: "1"
---
# DEFENSA DEL CONSUMIDOR

**DEFENSA DEL CONSUMIDOR**

**Ley 27266**

**Modificación. Ley N° 24.240.**

El Senado y Cámara de Diputados de la Nación Argentina reunidos en Congreso, etc. sancionan con fuerza de

Ley:

##### ARTÍCULO 1° — Sustitúyese el artículo 38 de la ley 24.240, que queda redactado de la siguiente manera:

##### Artícu

### Revisión de 10 documentos crudos — Etapa 0

Se seleccionaron diez documentos del corpus para inspeccionar su estructura, formato, metadatos y posibles dificultades de procesamiento.

La inspección permitió identificar los siguientes problemas:

**1. Heterogeneidad estructural.** Los documentos incluyen artículos, anexos, listados, tablas y secciones introductorias. Además, existen diferentes formas de representar los encabezados, como `Artículo`, `ARTÍCULO` y `Art.`.

**2. Artículos citados dentro de otras normas.** En `LEY-27266.md`, el artículo 1 reproduce el nuevo texto del artículo 38 de la Ley 24.240. Un extractor ingenuo podría confundir un artículo citado con una sección independiente del documento.

**3. Información sobre modificaciones y derogaciones.** En `DEC-21-2007.md` aparecen artículos con anotaciones explícitas de derogación. Esta información debe conservarse para evitar respuestas que confundan disposiciones históricas con disposiciones vigentes.

**4. Información tabular y anexos.** Documentos como `DEC-2453-1993.md` y `DEC-405-1998.md` contienen estructuras que podrían perder significado durante una limpieza demasiado agresiva.

**5. Relevancia temática variable.** Algunas normas regulan directamente aspectos de defensa del consumidor, mientras que otras mencionan organismos o funcionarios vinculados con esa temática. Esto puede introducir documentos poco relevantes en los resultados de recuperación.

**Conclusión:** El corpus ofrece información jurídica estructurada y aprovechable para construir un sistema RAG, pero requiere normalización, segmentación sensible a la estructura y validación de metadatos. En la Etapa 1 se deberá conservar la distinción entre artículos propios, textos citados, anexos y referencias a modificaciones normativas.

In [43]:
from collections import Counter
import re

# Identificador candidato: nombre del archivo sin extensión
ids = [path.stem for path in subconjunto]

conteo_ids = Counter(ids)
ids_duplicados = [
    identificador
    for identificador, cantidad in conteo_ids.items()
    if cantidad > 1
]

# Ejemplos de formatos reconocidos
patron_id = re.compile(
    r"^(LEY|DEC|DNU)-\d+(?:-\d{4})?$",
    re.IGNORECASE
)

ids_formato_no_reconocido = [
    identificador
    for identificador in ids
    if not patron_id.fullmatch(identificador)
]

print("--- AUDITORÍA DE IDENTIFICADORES ---")
print(f"Documentos: {len(subconjunto)}")
print(f"IDs únicos: {len(set(ids))}")
print(f"IDs duplicados: {len(ids_duplicados)}")
print(f"IDs con formato no reconocido: {len(ids_formato_no_reconocido)}")

print("\nIDs duplicados:")
print(ids_duplicados[:10])

print("\nIDs con formato no reconocido:")
print(ids_formato_no_reconocido[:10])

print("\nEjemplos de IDs:")
print(ids[:10])

--- AUDITORÍA DE IDENTIFICADORES ---
Documentos: 170
IDs únicos: 170
IDs duplicados: 0
IDs con formato no reconocido: 0

IDs duplicados:
[]

IDs con formato no reconocido:
[]

Ejemplos de IDs:
['DNU-271-2000', 'DEC-946-1989', 'LEY-27250', 'DNU-13-2015', 'DNU-1283-2003', 'DEC-991-2020', 'DEC-528-2011', 'DEC-276-1998', 'DNU-2156-2002', 'DEC-826-2011']


In [44]:
import re
from collections import Counter
from datetime import date

# Extraer los campos del encabezado YAML
def extraer_metadata(path):
    texto = path.read_text(encoding="utf-8")

    match = re.match(r"\A---\s*\n(.*?)\n---", texto, re.DOTALL)

    if not match:
        return {}

    metadata = {}

    for linea in match.group(1).splitlines():
        campo = re.match(r"^([\w-]+):\s*(.*)$", linea)

        if campo:
            clave = campo.group(1)
            valor = campo.group(2).strip().strip('"').strip("'")
            metadata[clave] = valor

    return metadata


# Validar fechas ISO (AAAA-MM-DD)
def fecha_valida(valor):
    if not valor or valor == "1900-01-01":
        return False

    try:
        date.fromisoformat(valor)
        return True
    except ValueError:
        return False


metadatos = [
    extraer_metadata(path)
    for path in subconjunto
]

total = len(metadatos)

# Contar fechas válidas y valores de estado
fechas_publicacion_validas = sum(
    fecha_valida(m.get("publication_date"))
    for m in metadatos
)

fechas_actualizacion_validas = sum(
    fecha_valida(m.get("last_updated"))
    for m in metadatos
)

estados = Counter(
    m.get("status", "<ausente>")
    for m in metadatos
)

# Revisar identificadores
identificadores = [
    m.get("identifier", "")
    for m in metadatos
]

ids_vacios = sum(not i for i in identificadores)
ids_duplicados = total - len(set(identificadores))

# Comparar identifier con nombre de archivo
inconsistencias = []

for path, m in zip(subconjunto, metadatos):
    if path.stem != m.get("identifier"):
        inconsistencias.append(
            (path.name, m.get("identifier"))
        )


print("--- CALIDAD DE METADATOS ---")

print(
    f"Fechas de publicación válidas: "
    f"{fechas_publicacion_validas}/{total} "
    f"({100 * fechas_publicacion_validas / total:.1f}%)"
)

print(
    f"Fechas last_updated válidas y no sospechosas: "
    f"{fechas_actualizacion_validas}/{total} "
    f"({100 * fechas_actualizacion_validas / total:.1f}%)"
)

print("\nValores encontrados en status:")
for estado, cantidad in estados.most_common():
    print(f"  {estado}: {cantidad}")

print("\nIdentificadores:")
print(f"IDs vacíos: {ids_vacios}")
print(f"IDs duplicados: {ids_duplicados}")
print(f"IDs distintos del nombre del archivo: {len(inconsistencias)}")
print("Ejemplos de diferencias:", inconsistencias[:5])

--- CALIDAD DE METADATOS ---
Fechas de publicación válidas: 170/170 (100.0%)
Fechas last_updated válidas y no sospechosas: 11/170 (6.5%)

Valores encontrados en status:
  in_force: 170

Identificadores:
IDs vacíos: 0
IDs duplicados: 0
IDs distintos del nombre del archivo: 0
Ejemplos de diferencias: []


## La auditoría, en una tabla

Las ocho preguntas del principio, respondidas para este corpus con los números de arriba. Tu
Etapa 0 completa la misma tabla para el tuyo -- y agrega una columna que esta deja en blanco a
propósito: **qué etapa posterior paga por cada "no".**

In [46]:

# RESULTADOS FINALES DE LA AUDITORÍA - ETAPA 0


# 1. Recuperar los resultados medidos en las celdas anteriores

total_tokens = int(df_validos["tokens"].sum())
promedio_tokens = float(df_validos["tokens"].mean())
mediana_tokens = float(df_validos["tokens"].median())
max_tokens = int(df_validos["tokens"].max())
min_tokens = int(df_validos["tokens"].min())

# Resultados del análisis de similitud Jaccard
max_similitud = max((par[2] for par in similitudes), default=0.0)
cantidad_casi_duplicados = len(pares_similares)

# Verificar que todos los documentos tengan conteo de tokens
assert len(df_validos) == len(subconjunto), (
    "Faltan documentos con conteo válido de tokens."
)


# 2. TABLA DE AUDITORÍA


properties = [
    (
        "1 answer in it",
        "Corpus jurídico en español, con normas relacionadas "
        "con defensa del consumidor. Cobertura de preguntas "
        "específicas pendiente de verificar.",
        "Etapa 2: construir preguntas con respuestas verificadas."
    ),
    (
        "2 clean text",
        "NO completamente. En la revisión de 10 documentos se "
        "observaron variaciones de encabezados, artículos citados, "
        "anexos, tablas y anotaciones de derogación.",
        "Etapa 1: normalización y limpieza que preserve "
        "la información jurídica."
    ),
    (
        "3 structure",
        "Existe estructura aprovechable, pero heterogénea: "
        "artículos, anexos, tablas y textos normativos citados. "
        "No alcanza con detectar únicamente encabezados de artículos.",
        "Etapa 1: segmentación estructural y validación "
        "de las secciones extraídas."
    ),
    (
        "4 stable ids",
        "Los 170 documentos tienen un identificador único y no vacío. "
        "En todos los casos coincide con el nombre del archivo. "
        "Falta comprobar su estabilidad entre descargas.",
        "Etapa 1: utilizar identifier como base de doc_id "
        "y validar su estabilidad."
    ),
    (
        "5 metadata",
        "Cobertura del 100% en publication_date, last_updated y status. "
        "Las 170 fechas de publicación tienen formato válido, "
        "pero 159 documentos contienen last_updated = 1900-01-01. "
        "Todos declaran status = in_force, sin vigencia verificada.",
        "Etapa 1: validar fechas y vigencia, "
        "tratar las fechas sospechosas y mapear status "
        "al esquema de metadatos."
    ),
    (
        "6 changes",
        "El corpus contiene legislación susceptible de reformas. "
        "Falta identificar y verificar las relaciones entre versiones.",
        "Etapa 1: registrar versiones y relaciones entre normas."
    ),
    (
        "7 eval set",
        "Se realizaron cinco preguntas a libro cerrado. "
        "El corpus permite plantear una evaluación jurídica, "
        "pero falta construir las etiquetas gold.",
        "Etapa 2: construir y congelar el conjunto de evaluación."
    ),
    (
        "8 similarity",
        f"Muestra de {len(muestra)} documentos y "
        f"{len(similitudes)} pares. "
        f"Similitud Jaccard máxima: {max_similitud:.3f}. "
        f"Pares con Jaccard >= {UMBRAL}: "
        f"{cantidad_casi_duplicados}.",
        "Etapa 1: inspeccionar los pares más similares."
    ),
]

print_table(
    "legalize-ar (Defensa del Consumidor): la auditoría",
    ["medido", "etapa posterior"],
    [
        (propiedad, {
            "medido": resultado,
            "etapa posterior": etapa
        })
        for propiedad, resultado, etapa in properties
    ],
    smoke=SMOKE_TEST
)



# 3. GUARDAR RESULTADOS DE LA AUDITORÍA


store.write_json(ART / "stage0_audit.json", {
    "profile": PROFILE,
    "corpus": "legalize-ar",
    "tema": "Defensa del Consumidor",

    "archivos": len(subconjunto),

    "datos_sucios": {
           "fechas_malas_1900": fechas_malas,
          "variaciones_articulo": sorted(encabezados_encontrados)[:5],
          "publication_date_formato_valido": fechas_publicacion_validas,
          "last_updated_no_sospechoso": fechas_actualizacion_validas,
          "status_in_force": estados.get("in_force", 0),
          "ids_vacios": ids_vacios,
          "ids_duplicados": ids_duplicados,
          "ids_distintos_nombre_archivo": len(inconsistencias)
    },

    "size": {
        "characters": total_chars,
        "tokenizer": "Qwen/Qwen3-8B",
        "tokens_total": total_tokens,
        "tokens_promedio": promedio_tokens,
        "tokens_mediana": mediana_tokens,
        "tokens_maximo": max_tokens,
        "tokens_minimo": min_tokens,
        "num_ctx": NUM_CTX,
        "ventanas_equivalentes": total_tokens / NUM_CTX
    },

    "similarity": {
        "metodo": "Jaccard de shingles de 5 palabras",
        "semilla": 42,
        "documentos_muestra": len(muestra),
        "pares_comparados": len(similitudes),
        "umbral": UMBRAL,
        "pares_sobre_umbral": cantidad_casi_duplicados,
        "similitud_maxima": max_similitud,
        "top_10": [
            {
                "documento_1": doc1,
                "documento_2": doc2,
                "jaccard": score
            }
            for doc1, doc2, score in similitudes[:10]
        ]
    }
})



# 4. REGISTRAR Y FINALIZAR LA EJECUCIÓN


run.note("tokens_total", total_tokens)
run.note("documentos", len(subconjunto))
run.note("similitud_maxima", max_similitud)

run.finish()

print("Auditoría de la Etapa 0 guardada correctamente.")


legalize-ar (Defensa del Consumidor): la auditoría
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
               | medido                                                                                                                                                                                                                                  | etapa posterior                                                                                           
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [49]:

# Crear la ficha del corpus con los resultados de la Etapa 0

total_tokens = int(df_validos["tokens"].sum())
ventanas_equivalentes = total_tokens / NUM_CTX

par_mas_similar = similitudes[0]
doc1, doc2, similitud_maxima = par_mas_similar

contenido = f"""# Ficha del Corpus: Legislación Argentina (Defensa del Consumidor)

## Fuente y selección
- **Fuente y versión fijada:** Repositorio `legalize-ar`, commit `3d2450b`.
- **Licencia y atribución:** Pendientes de verificar en la fuente original.
- **Regla del subconjunto:** 170 documentos relacionados con defensa del consumidor, seleccionados mediante las palabras clave "defensa del consumidor", "24.240" y "ley 24240".
- **Idioma:** Español.
- **Formato original:** Markdown con metadatos YAML.

## Tamaño del corpus
- **Documentos analizados:** {len(subconjunto)}.
- **Tokens totales:** {total_tokens:,} (tokenizer `Qwen/Qwen3-8B`).
- **Promedio de tokens por documento:** {df_validos["tokens"].mean():,.1f}.
- **Mediana de tokens por documento:** {df_validos["tokens"].median():,.1f}.
- **Documento más largo:** {df_validos["tokens"].max():,} tokens.
- **Ventana del generador:** {NUM_CTX} tokens.
- **Equivalencia aproximada:** {ventanas_equivalentes:.1f} ventanas.

## Calidad de metadatos
- Los 170 documentos contienen `identifier`, `publication_date`, `last_updated` y `status`.
- **Fechas de publicación con formato válido:** {fechas_publicacion_validas}/{len(subconjunto)}.
- **Fechas de actualización sospechosas:** {fechas_malas}/{len(subconjunto)} documentos contienen `last_updated = 1900-01-01`.
- **Fechas de actualización no sospechosas y con formato válido:** {fechas_actualizacion_validas}/{len(subconjunto)}.
- **Estado declarado:** {estados.get("in_force", 0)} documentos tienen `status = in_force`, sin verificación individual de vigencia.
- **Identificadores:** {len(subconjunto) - ids_vacios} identificadores no vacíos y {ids_duplicados} duplicados.

## Defectos conocidos
- Variaciones tipográficas en los encabezados de artículos (`Artículo`, `ARTÍCULO`, `Art.`).
- Artículos de otras normas reproducidos dentro de leyes modificatorias.
- Anotaciones sobre derogaciones y modificaciones normativas.
- Documentos con anexos, tablas y listados que requieren preservar su estructura.
- Diferencias importantes de extensión entre documentos.
- Documentos con relevancia temática indirecta.
- Metadatos de actualización sospechosos y vigencia no verificada.

## Similitud entre documentos
- **Método:** Jaccard sobre shingles de cinco palabras.
- **Muestra:** {len(muestra)} documentos, con semilla 42.
- **Pares comparados:** {len(similitudes)}.
- **Similitud máxima:** {similitud_maxima:.3f}.
- **Par más similar:** `{doc1}` y `{doc2}`.
- **Umbral exploratorio:** {UMBRAL}.
- **Pares que alcanzan el umbral:** {len(pares_similares)}.

Los resultados corresponden a una muestra y no descartan
documentos similares fuera de ella.

## Qué permite evaluar este corpus
- Recuperación de artículos y disposiciones jurídicas específicas.
- Identificación de normas relevantes frente a menciones indirectas.
- Recuperación de información distribuida entre artículos y anexos.
- Tratamiento de modificaciones y derogaciones.
- Respuestas fundamentadas dentro de un presupuesto de contexto limitado.

## Limitaciones pendientes
- Verificar la estabilidad de los identificadores entre descargas.
- Contrastar las fechas y los estados de vigencia con fuentes oficiales.
- Verificar la licencia y atribución de la fuente.
- Validar la extracción de artículos, anexos y textos normativos citados.
"""

with open("corpus_card.md", "w", encoding="utf-8") as f:
    f.write(contenido)

print("¡Archivo corpus_card.md actualizado correctamente!")


¡Archivo corpus_card.md actualizado correctamente!
